# Runner Onchain (1d, 4 symbols)

Pipeline:
1. Config
2. WF run via `run_runner_onchain.py` (subprocess)
3. Reports, tables, plots
4. Quick sanity


In [1]:
# ==== Cell 1: Setup + Config ====
import logging
import subprocess
import sys
from pathlib import Path

import pandas as pd

try:
    from IPython.display import display
except Exception:
    display = print

from src.artifacts import safe_write_parquet
from src import report as report_mod

logging.basicConfig(level=logging.INFO, format="%(asctime)s | %(levelname)s | %(message)s")
logger = logging.getLogger("runner_onchain_nb")

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "src").exists() and (PROJECT_ROOT.parent / "src").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
if not (PROJECT_ROOT / "src").exists():
    raise FileNotFoundError(f"Cannot locate project root from cwd={Path.cwd()}")

SCRIPT_PATH = PROJECT_ROOT / "run_runner_onchain.py"
RESULTS_DIR = PROJECT_ROOT / "results" / "runner_onchain"
FIGURES_DIR = PROJECT_ROOT / "figures" / "runner_onchain"
PPY = 365
TOP_K_PLOTS = 3

if not SCRIPT_PATH.exists():
    raise FileNotFoundError(f"Missing script: {SCRIPT_PATH}")

RESULTS_DIR.mkdir(parents=True, exist_ok=True)
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

print("PROJECT_ROOT:", PROJECT_ROOT)
print("SCRIPT_PATH:", SCRIPT_PATH)
print("RESULTS_DIR:", RESULTS_DIR)
print("FIGURES_DIR:", FIGURES_DIR)


PROJECT_ROOT: c:\Users\prosh\OneDrive\Рабочий стол\git\diploma
SCRIPT_PATH: c:\Users\prosh\OneDrive\Рабочий стол\git\diploma\run_runner_onchain.py
RESULTS_DIR: c:\Users\prosh\OneDrive\Рабочий стол\git\diploma\results\runner_onchain
FIGURES_DIR: c:\Users\prosh\OneDrive\Рабочий стол\git\diploma\figures\runner_onchain


In [2]:
# ==== Cell 2: WF Runs via Script ====
import time

cmd = [sys.executable, "-W", "ignore::UserWarning", "-u", str(SCRIPT_PATH), "--mp-workers", "4"]  # -u = unbuffered stdout
print("Running:", " ".join(cmd))

t0 = time.time()
proc = subprocess.Popen(
    cmd,
    cwd=str(PROJECT_ROOT),
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1,
)

for line in proc.stdout:
    print(line, end="")

rc = proc.wait()
print(f"\nReturn code: {rc} | elapsed: {(time.time() - t0)/60:.1f} min")

if rc != 0:
    raise RuntimeError(f"WF script failed with return code={rc}")


Running: c:\Users\prosh\OneDrive\Рабочий стол\git\diploma\.venv\Scripts\python.exe -W ignore::UserWarning -u c:\Users\prosh\OneDrive\Рабочий стол\git\diploma\run_runner_onchain.py --mp-workers 4
2026-04-05 18:20:04,466 | INFO | Configured backtesting.Pool = multiprocessing.Pool(processes=4).
2026-04-05 18:20:04,467 | INFO | PROJECT_ROOT=c:\Users\prosh\OneDrive\Р Р°Р±РѕС‡РёР№ СЃС‚РѕР»\git\diploma
2026-04-05 18:20:04,476 | INFO | On-chain strategies: ['R1OC:RSI_MR_Onchain', 'R2OC:Bollinger_MR_Onchain', 'R3OC:ZScore_MR_Onchain', 'S1OC:MAFilter_RSI_MR_Onchain', 'S2OC:MA200Filter_Bollinger_MR_Onchain']
2026-04-05 18:20:04,476 | INFO | Benchmark: BENCH:BuyHold
2026-04-05 18:20:04,522 | INFO | BTCUSDT loaded: n=3088 | 2017-09-16 .. 2026-02-28
2026-04-05 18:20:04,558 | INFO | ETHUSDT loaded: n=3088 | 2017-09-16 .. 2026-02-28
2026-04-05 18:20:04,589 | INFO | XRPUSDT loaded: n=2828 | 2018-06-03 .. 2026-02-28
2026-04-05 18:20:04,617 | INFO | DOGEUSDT loaded: n=2401 | 2019-08-04 .. 2026-02-28
2026

In [3]:
# ==== Cell 3: Reports, Tables, and Plots ====
folds_best = pd.read_parquet(RESULTS_DIR / "folds_best.parquet")
returns_oos = pd.read_parquet(RESULTS_DIR / "returns_oos.parquet")
bench_oos = pd.read_parquet(RESULTS_DIR / "bench_returns_oos.parquet")
bench_folds = pd.read_parquet(RESULTS_DIR / "bench_folds.parquet")


def _ensure_date_col(df_in: pd.DataFrame) -> pd.DataFrame:
    df = df_in.copy()
    if "date" not in df.columns:
        if "datetime_utc" in df.columns:
            df = df.rename(columns={"datetime_utc": "date"})
        elif "index" in df.columns:
            df = df.rename(columns={"index": "date"})
        else:
            raise KeyError(f"No date column in dataframe. Columns={list(df.columns)[:30]}")
    df["date"] = pd.to_datetime(df["date"])
    return df


def _build_benchmark_comparison(folds_best_df: pd.DataFrame, bench_folds_df: pd.DataFrame):
    if bench_folds_df.empty or folds_best_df.empty:
        return pd.DataFrame(), pd.DataFrame()

    merged = folds_best_df.merge(
        bench_folds_df[["symbol", "cost", "fold_id", "bench_calmar", "bench_total_return"]],
        on=["symbol", "cost", "fold_id"],
        how="left",
        validate="many_to_one",
    )
    merged["beat_bench_calmar"] = (merged["oos_calmar"] > merged["bench_calmar"]).astype(int)
    merged["beat_bench_total_return"] = (merged["oos_total_return"] > merged["bench_total_return"]).astype(int)
    merged["excess_calmar"] = merged["oos_calmar"] - merged["bench_calmar"]
    merged["excess_total_return"] = merged["oos_total_return"] - merged["bench_total_return"]

    beat = (
        merged.groupby(["symbol", "cost", "strategy_id"])
        .agg(
            folds=("fold_id", "nunique"),
            beat_calmar_share=("beat_bench_calmar", "mean"),
            beat_return_share=("beat_bench_total_return", "mean"),
            calmar_median=("oos_calmar", "median"),
            calmar_worst=("oos_calmar", "min"),
        )
        .reset_index()
    )
    excess = (
        merged.groupby(["symbol", "cost", "strategy_id"])
        .agg(
            folds=("fold_id", "nunique"),
            excess_calmar_median=("excess_calmar", "median"),
            excess_return_median=("excess_total_return", "median"),
        )
        .reset_index()
    )
    return beat, excess


def _plot_top_equity_drawdown(returns_oos_df: pd.DataFrame, bench_oos_df: pd.DataFrame, stitched_df: pd.DataFrame, top_k: int):
    rets = _ensure_date_col(returns_oos_df)
    bench = _ensure_date_col(bench_oos_df)

    if stitched_df.empty:
        return

    ranked = stitched_df.sort_values(["symbol", "cost", "Calmar"], ascending=[True, True, False])
    bench_id = "BENCH:BuyHold"
    for (sym, cost), group in ranked.groupby(["symbol", "cost"]):
        top = group.head(int(top_k))
        for _, row in top.iterrows():
            sid = str(row["strategy_id"])
            if sid.startswith("BENCH:"):
                continue

            r = rets[(rets["symbol"] == sym) & (rets["cost"] == cost) & (rets["strategy_id"] == sid)].copy()
            b = bench[(bench["symbol"] == sym) & (bench["cost"] == cost) & (bench["strategy_id"] == bench_id)].copy()
            if r.empty or b.empty:
                continue

            rs = r.set_index("date")["ret"].astype(float)
            bs = b.set_index("date")["ret"].astype(float)
            report_mod.plot_equity_and_drawdown(
                returns=rs,
                bench_returns=bs,
                title=f"{sid} vs Buy&Hold | {sym} | {cost}",
                out_path=FIGURES_DIR / f"equity_dd__{sym}__{cost}__{sid.replace(':', '_')}.png",
            )


leaderboard = report_mod.leaderboard_by_strategy(folds_best)
stitched = report_mod.stitched_metrics(returns_oos, periods_per_year=PPY)
bench_stitched = report_mod.stitched_metrics(bench_oos, periods_per_year=PPY)
beat, excess = _build_benchmark_comparison(folds_best, bench_folds)

safe_write_parquet(leaderboard, RESULTS_DIR / "leaderboard.parquet")
safe_write_parquet(stitched, RESULTS_DIR / "stitched_metrics.parquet")
safe_write_parquet(bench_stitched, RESULTS_DIR / "bench_stitched_metrics.parquet")
safe_write_parquet(beat, RESULTS_DIR / "beat_benchmark_summary.parquet")
safe_write_parquet(excess, RESULTS_DIR / "excess_vs_benchmark_summary.parquet")

_plot_top_equity_drawdown(returns_oos, bench_oos, stitched, TOP_K_PLOTS)

print("Saved report artifacts to", RESULTS_DIR)
print("Saved figures to", FIGURES_DIR)


Saved report artifacts to c:\Users\prosh\OneDrive\Рабочий стол\git\diploma\results\runner_onchain
Saved figures to c:\Users\prosh\OneDrive\Рабочий стол\git\diploma\figures\runner_onchain


In [4]:
# ==== Cell 4: Quick sanity view ====


def _shape(path: Path):
    if not path.exists():
        return None
    return pd.read_parquet(path).shape

keys = [
    "folds_best.parquet",
    "returns_oos.parquet",
    "bench_returns_oos.parquet",
    "bench_folds.parquet",
    "leaderboard.parquet",
    "stitched_metrics.parquet",
    "bench_stitched_metrics.parquet",
    "beat_benchmark_summary.parquet",
    "excess_vs_benchmark_summary.parquet",
]

shape_df = pd.DataFrame(
    [{"file": k, "shape": _shape(RESULTS_DIR / k)} for k in keys]
)
display(shape_df)

display(pd.read_parquet(RESULTS_DIR / "stitched_metrics.parquet").head(20))


,file,shape
0,folds_best.parquet,"(540, 23)"
1,returns_oos.parquet,"(98640, 5)"
2,bench_returns_oos.parquet,"(19728, 5)"
3,bench_folds.parquet,"(108, 13)"
4,leaderboard.parquet,"(60, 36)"
5,stitched_metrics.parquet,"(60, 12)"
6,bench_stitched_metrics.parquet,"(12, 12)"
7,beat_benchmark_summary.parquet,"(60, 8)"
8,excess_vs_benchmark_summary.parquet,"(60, 6)"


,symbol,cost,strategy_id,n_bars,Total Return,CAGR,Ann. Vol,Sharpe,Sortino,MaxDD,Calmar,CDaR_95
0,BTCUSDT,Base,S1OC:MAFilter_RSI_MR_Onchain,1826,0.238999,0.043768,0.157493,0.350848,0.509252,-0.227380,0.192488,0.151384
1,BTCUSDT,Base,S2OC:MA200Filter_Bollinger_MR_Onchain,1826,0.173343,0.032470,0.163633,0.277046,0.402107,-0.227380,0.142800,0.151048
2,BTCUSDT,Base,R2OC:Bollinger_MR_Onchain,1826,0.190268,0.035430,0.280206,0.264861,0.388306,-0.375023,0.094474,0.320331
3,BTCUSDT,Base,R3OC:ZScore_MR_Onchain,1826,0.139822,0.026505,0.263473,0.231836,0.334114,-0.375023,0.070676,0.328694
4,BTCUSDT,Base,R1OC:RSI_MR_Onchain,1826,-0.114573,-0.024030,0.282272,0.055389,0.080485,-0.447579,-0.053689,0.405006
5,BTCUSDT,High,S1OC:MAFilter_RSI_MR_Onchain,1826,0.219822,0.040519,0.157494,0.331063,0.479435,-0.227380,0.178197,0.151796
6,BTCUSDT,High,S2OC:MA200Filter_Bollinger_MR_Onchain,1826,0.152183,0.028721,0.163623,0.254831,0.369146,-0.227380,0.126312,0.151477
7,BTCUSDT,High,R2OC:Bollinger_MR_Onchain,1826,0.158216,0.029795,0.280257,0.245413,0.359393,-0.379879,0.078434,0.326830
8,BTCUSDT,High,R3OC:ZScore_MR_Onchain,1826,0.110571,0.021185,0.263525,0.212145,0.305313,-0.379879,0.055767,0.335447
9,BTCUSDT,High,R1OC:RSI_MR_Onchain,1826,-0.141340,-0.030000,0.282513,0.033896,0.049180,-0.457270,-0.065608,0.415824
